In [1]:
import functools
import os
import sys
import traceback
from typing import Dict, Literal, Optional, Tuple
import cellflow
import scanpy as sc
import numpy as np
import functools
from ott.solvers import utils as solver_utils
import optax
from omegaconf import OmegaConf
from typing import NamedTuple, Any
import hydra
import wandb
import anndata as ad
import pandas as pd
import os
from cellflow.training import ComputationCallback
from cellflow.preprocessing import transfer_labels, compute_wknn
from cellflow.training import ComputationCallback
from numpy.typing import ArrayLike
from cellflow.metrics import compute_r_squared, compute_e_distance
from cellflow.metrics import compute_r_squared, compute_e_distance, compute_scalar_mmd, compute_sinkhorn_div
import sys
import pickle
from cellflow.preprocessing import transfer_labels, compute_wknn, centered_pca, project_pca





/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/optuna/study/_optimize.py:29: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from optuna import progress_bar as pbar_module
/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/anndata/utils.py:429: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  warnings.warn(msg, FutureWa

In [2]:
def compute_metrics(adata_ref: ad.AnnData, adata_pred: ad.AnnData, deg_dict: dict, adata_ood_true: ad.AnnData, adata_ctrl: ad.AnnData, n_neighbors: int=1, cell_type_col: str = "cell_type_new", min_cells_for_dist_metrics: int = 50) -> dict:
    compute_wknn(ref_adata=adata_ref, query_adata=adata_pred, n_neighbors=n_neighbors, ref_rep_key="X_pca", query_rep_key="X_pca_for_ct_transfer")
    transfer_labels(query_adata=adata_pred, ref_adata=adata_ref, label_key=cell_type_col)

    deg_r_sq = {}
    for k in deg_genes.keys():
        donor_deg_dict = {k: v for k, v in deg_genes[k].items() if (k.startswith(donor_held_out) and k.endswith(f"_{cytokine}"))}
        deg_r_sq[k] = {}
        for ct_cyto in donor_deg_dict.keys():
            cell_type = ct_cyto.split("_")[1]
            adata_true_ct = adata_ood_true[(adata_ood_true.obs[f"{cell_type_col}"]==cell_type)]
            adata_pred_ct = adata_pred[adata_pred.obs[f"{cell_type_col}_transfer"]==cell_type]
            if adata_pred_ct.n_obs == 0:
                continue
        
            deg_mask = [True if el in donor_deg_dict[ct_cyto] else False for el in adata_ood_true.var_names]
            deg_true_decoded = adata_true_ct[:,deg_mask].X.toarray()
            deg_pred_decoded = adata_pred_ct[:,deg_mask].X
            deg_r_sq[k][f"deg_decoded_r_squared_{cell_type}"] = compute_r_squared(deg_true_decoded, deg_pred_decoded)
        
      
    return deg_r_sq


def mean_model_1(adata_train, adata_ctrl_current_donor, donor):
    # Mean displacement of same donor across cytokines
    # i.e. this is a constant prediction for all test cytokines
    adata_perturbed_same_donor = adata_train[(adata_train.obs["donor"]==donor) & (adata_train.obs["cytokine"]!="PBS")]
    control_mean = adata_ctrl_current_donor.X.mean(axis=0)
    displacement_vecs = []
    for cyto in adata_perturbed_same_donor.obs["cytokine"].unique():
        displacement_vecs.append(np.asarray(adata_perturbed_same_donor[adata_perturbed_same_donor.obs["cytokine"]==cyto].X.mean(axis=0) - control_mean).squeeze())
    displacement = np.array(displacement_vecs).mean(axis=0)
    return np.asarray(adata_ctrl_current_donor.X.toarray() + displacement)

def mean_model_2(adata_train, adata_ctrl_current_donor, cytokine):
    # Mean displacement of same cytokine across donors
    adata_perturbed_same_cytokine = adata_train[adata_train.obs["cytokine"]==cytokine]
    control_mean = adata_ctrl_current_donor.X.mean(axis=0)
    displacement_vecs = []
    for donor in adata_perturbed_same_cytokine.obs["donor"].unique():
        displacement_vecs.append(np.asarray(adata_perturbed_same_cytokine[adata_perturbed_same_cytokine.obs["donor"]==donor].X.mean(axis=0) - control_mean).squeeze())
    displacement = np.array(displacement_vecs).mean(axis=0)
    return np.asarray(adata_ctrl_current_donor.X.toarray() + displacement)


In [24]:
out_dir = "/lustre/groups/ml01/workspace/ot_perturbation/models/additive_model/pbmc_new_donor/identity"
donor_held_out = "Donor1"
idx_given_donor = "15"

control_key = "is_control"

    
adata_train = sc.read_h5ad(f"/lustre/groups/ml01/workspace/ot_perturbation/data/pbmc/new_donor/{donor_held_out}/{str(idx_given_donor)}/adata_train_{donor_held_out}.h5ad")
adata_ood_perturbed  = sc.read_h5ad(f"/lustre/groups/ml01/workspace/ot_perturbation/data/pbmc/new_donor/{donor_held_out}/{str(idx_given_donor)}/adata_ood_{donor_held_out}.h5ad")
cytokines_to_impute = adata_train.uns["split_info"][idx_given_donor]["cytokines_to_impute"]
cytokines_to_train_data = adata_train.uns["split_info"][idx_given_donor]["cytokines_to_train_data"]
if len(cytokines_to_train_data) != 65:
    sys.exit(0)

adata_ctrl = adata_train[adata_train.obs[control_key].to_numpy()]


with open("/lustre/groups/ml01/workspace/ot_perturbation/data/pbmc/idcs_to_keep.pkl", "rb") as pickle_file:
    idcs_to_keep = pickle.load(pickle_file)
adata_full = sc.read_h5ad("/lustre/groups/ml01/workspace/ot_perturbation/data/pbmc/pbmc_with_pca.h5ad")
adata_ref = adata_full[adata_full.obs_names.isin(idcs_to_keep)]

with open("/lustre/groups/ml01/workspace/ot_perturbation/data/pbmc/degs_different_top_k.pkl", "rb") as pickle_file:
    deg_genes = pickle.load(pickle_file)

adata_ctrl_current_donor = adata_ctrl[adata_ctrl.obs["donor"]==donor_held_out]
if adata_ctrl_current_donor.n_obs > 10000:
        sc.pp.subsample(adata_ctrl_current_donor, n_obs=10000)

for cytokine in cytokines_to_impute:
    pred1 = adata_ctrl_current_donor.X.toarray()
    condition = f"{donor_held_out}_{cytokine}"
    conditions = [condition] * pred1.shape[0]

    obs_data = pd.DataFrame({
        'condition': conditions
    })

    adata_pred = ad.AnnData(X=pred1, obs=obs_data)
    adata_pred.obs["cytokine"] = cytokine
    adata_pred.obs["donor"] = donor_held_out
    adata_pred.var_names=adata_ctrl.var_names   
    
    project_pca(query_adata=adata_pred, ref_adata=adata_ref, obsm_key_added="X_pca_for_ct_transfer")
    project_pca(query_adata=adata_pred, ref_adata=adata_full, obsm_key_added="X_pca")
    cond_orig = condition
    condition = condition + "_" + str(len(cytokines_to_train_data))
    adata_ood_true = adata_full[(adata_full.obs["donor"] == donor_held_out) & (adata_full.obs["cytokine"]==cytokine)]
    
    out = compute_metrics(adata_ref=adata_ref, adata_pred=adata_pred, deg_dict=deg_genes, adata_ood_true=adata_ood_true, adata_ctrl=adata_ctrl_current_donor)
    df = pd.DataFrame.from_dict(out)
    df["condition"]=condition
    df["num_cytokines_in_train"] = len(cytokines_to_train_data)
    
    df.to_csv(os.path.join(out_dir, f"{idx_given_donor}_{condition}.csv"))



/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/ictstr01/home/icb/dominik.klein/git_repos/cell_flow_perturbation/src/cellflow/preprocessing/_wknn.py:88: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.
  ref_adata.uns[uns_key_added] = wknn
/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/home/icb/dominik.klein/mambaforge/envs/cellflow/lib/python3.12/site-packages/anndata/_core/aligned_df.py:68: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/home/icb/dominik.klein/mambafo

In [4]:
adata_ref

View of AnnData object with n_obs × n_vars = 203117 × 2000
    obs: 'sample', 'species', 'gene_count', 'tscp_count', 'mread_count', 'bc1_wind', 'bc2_wind', 'bc3_wind', 'bc1_well', 'bc2_well', 'bc3_well', 'log1p_n_genes_by_counts', 'log1p_total_counts', 'total_counts_MT', 'pct_counts_MT', 'log1p_total_counts_MT', 'donor', 'cytokine', 'treatment', 'cell_type', 'condition', 'is_control', 'cell_type_new', 'donor_cell_type'
    var: 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'donor_one_hot', 'esm2_embeddings', 'hvg', 'log1p', 'pca'
    obsm: 'X_pca'
    varm: 'PCs', 'X_mean'
    layers: 'counts'

In [5]:
adata_pred

AnnData object with n_obs × n_vars = 10000 × 2000
    obs: 'condition', 'cytokine', 'donor'
    obsm: 'X_pca_for_ct_transfer', 'X_pca'

In [6]:
adata_ood_true

View of AnnData object with n_obs × n_vars = 15246 × 2000
    obs: 'sample', 'species', 'gene_count', 'tscp_count', 'mread_count', 'bc1_wind', 'bc2_wind', 'bc3_wind', 'bc1_well', 'bc2_well', 'bc3_well', 'log1p_n_genes_by_counts', 'log1p_total_counts', 'total_counts_MT', 'pct_counts_MT', 'log1p_total_counts_MT', 'donor', 'cytokine', 'treatment', 'cell_type', 'condition', 'is_control', 'cell_type_new', 'donor_cell_type'
    var: 'n_cells', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'donor_one_hot', 'esm2_embeddings', 'hvg', 'log1p', 'pca'
    obsm: 'X_pca'
    varm: 'PCs', 'X_mean'
    layers: 'counts'

In [7]:
adata_ctrl_current_donor

AnnData object with n_obs × n_vars = 10000 × 2000
    obs: 'sample', 'species', 'gene_count', 'tscp_count', 'mread_count', 'bc1_wind', 'bc2_wind', 'bc3_wind', 'bc1_well', 'bc2_well', 'bc3_well', 'log1p_n_genes_by_counts', 'log1p_total_counts', 'total_counts_MT', 'pct_counts_MT', 'log1p_total_counts_MT', 'donor', 'cytokine', 'treatment', 'cell_type', 'condition', 'is_control'
    uns: 'cytokines_to_impute', 'cytokines_to_train_data', 'donor_embeddings', 'esm2_embeddings', 'hvg', 'log1p', 'split_info'
    layers: 'counts'

In [8]:
adata_ref.obsm["X_pca"].shape

(203117, 30)

In [13]:
adata_pred.obsm["X_pca_for_ct_transfer"]

array([[nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       ...,
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan]])

In [16]:
np.isnan(adata_ref.obsm["X_pca"]).sum()

0

In [17]:
project_pca(query_adata=adata_pred, ref_adata=adata_ref, obsm_key_added="X_pca_for_ct_transfer")
    

In [20]:
np.isnan(adata_pred.obsm["X_pca_for_ct_transfer"]).sum()

300000

In [22]:
pred1

array([[nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       ...,
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan]], dtype=float32)